# Find the Largest Learning Rate That Does Not Explode

**Topic:** LR range test &nbsp;·&nbsp; **Difficulty:** Medium

There is a largest stable learning rate, and for a quadratic you can write it
down: `2 / L`, where `L` is the biggest eigenvalue of the Hessian. Above it,
gradient descent diverges no matter how good everything else is.

Verify the bound empirically with an LR range test: sweep the rate, run a short
burst at each, and find where the loss stops falling and starts exploding.

## The data (created in the next cell)

- A quadratic with diagonal Hessian `A`, largest eigenvalue `L`.
- `lrs`: 60 candidate learning rates, log-spaced.
- 200 steps per candidate from the same start.

## Your task

With **NumPy only**:

1. The loss is `f(w) = 0.5 * w @ (A @ w)`, so each step is
   `w = w - lr * (A @ w)`. For each learning rate, run 200 of these steps
   from `w0` and record the final loss. If the run overflows or the loss
   exceeds `1e30`, record `1e30` (a finite sentinel, so the array stays
   comparable).
2. `lr_best`: the rate with the lowest final loss.
3. `lr_max_stable`: the largest rate whose final loss is still finite and below
   the starting loss.
4. `theory_bound`: `2 / L`.
5. Set `answers` to a dict with `"final_losses"` (length-60), `"lr_best"`,
   `"lr_max_stable"`, `"theory_bound"`.

## Evaluation

Compared to the reference to a relative tolerance of 1e-6. The evaluation cell
checks the empirical stability limit sits just below the theoretical `2/L`.

In [ ]:
import numpy as np

rng = np.random.default_rng(2323)
d = 30
eig = np.linspace(0.5, 40.0, d)
A = np.diag(eig)
L = float(eig.max())
w0 = rng.normal(0, 1, d)
lrs = np.logspace(-3, np.log10(0.2), 60)
n_steps = 200
print(f"largest eigenvalue L = {L:.2f}, so the theoretical limit is 2/L = {2/L:.4f}")

In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.

In [ ]:
# =====================================================================
# EVALUATION: run this cell to score your answer
# =====================================================================
import numpy as np

if "answers" not in dir():
    raise NameError("Define `answers` in the solution cell before running this cell.")
if not isinstance(answers, dict):
    raise TypeError("`answers` must be a dict.")

def _digest(a):
    a = np.asarray(a, dtype=float).ravel()
    if a.size == 0:
        return 0.0
    probe = np.random.default_rng(9176).standard_normal(a.size)
    return float(a @ probe)

_CHECKS = [
    ("final_losses", [60], 1.5000000000000004e+31, -4.2956421207734074e+30, 1e-06),
    ("lr_best", [], 0.04753591536752133, -0.007959319301527814, 1e-06),
    ("lr_max_stable", [], 0.04753591536752133, -0.007959319301527814, 1e-06),
    ("theory_bound", [], 0.05, -0.008371900740724959, 1e-06),
]

print("=" * 56)
_ok = True
for _key, _shape, _sum, _probe, _tol in _CHECKS:
    if _key not in answers:
        print(f"  {_key:<24} MISSING")
        _ok = False
        continue
    _a = np.asarray(answers[_key], dtype=float)
    if list(_a.shape) != _shape:
        print(f"  {_key:<24} shape {list(_a.shape)}, expected {_shape}")
        _ok = False
        continue
    if not np.all(np.isfinite(_a)):
        print(f"  {_key:<24} contains NaN or inf")
        _ok = False
        continue
    _g = (abs(_a.sum() - _sum) <= _tol * (1.0 + abs(_sum))
          and abs(_digest(_a) - _probe) <= _tol * (1.0 + abs(_probe)))
    print(f"  {_key:<24} {'ok' if _g else 'wrong'}")
    _ok = _ok and _g

if "lr_max_stable" in answers:
    _g = answers["lr_max_stable"] <= answers["theory_bound"]
    print(f"  {'empirical <= 2/L':<24} {'ok' if _g else 'wrong'}  "
          f"({answers['lr_max_stable']:.4f} vs {answers['theory_bound']:.4f})")
    _ok = _ok and _g

print("=" * 56)
if _ok:
    print("PASSED, nice work!")
else:
    print("NOT PASSED yet, revisit your solution and try again.")

## Solution

For `f(w) = 0.5 w^T A w`, a gradient step multiplies each eigen-coordinate by
`1 - lr * lambda_i`. That factor has magnitude below 1 exactly when
`lr < 2 / lambda_i`, so the binding constraint is the largest eigenvalue and the
stability limit is `2 / L`. The rate that converges fastest in theory is
`2 / (L + lambda_min)`, here `2 / 40.5 = 0.0494`, which sits between the largest
stable point on this grid and the first unstable one, so on this sweep `lr_best`
and `lr_max_stable` coincide. Convergence is still governed by the *worst*
contraction factor across coordinates, which is why ill-conditioned problems
are slow at every learning rate.

```python
import numpy as np

def run(lr):
    w = w0.copy()
    for _ in range(n_steps):
        w = w - lr * (A @ w)
        if not np.all(np.isfinite(w)):
            return 1e30
    with np.errstate(over="ignore", invalid="ignore"):
        val = 0.5 * w @ (A @ w)
    return float(min(val, 1e30)) if np.isfinite(val) else 1e30

final_losses = np.array([run(lr) for lr in lrs])
start_loss = float(0.5 * w0 @ (A @ w0))

lr_best = float(lrs[int(np.argmin(final_losses))])
stable = (final_losses < 1e30) & (final_losses < start_loss)
lr_max_stable = float(lrs[np.flatnonzero(stable).max()])

answers = {
    "final_losses": final_losses,
    "lr_best": lr_best,
    "lr_max_stable": lr_max_stable,
    "theory_bound": float(2.0 / L),
}
```